# 9.2 · 神经网络从零实现 / Neural Network from Scratch (NumPy)

> **课程定位 / Where this fits**
> 第 2 课，**Part 9 · 深度学习基础**。
> Lesson 2, **Part 9 · Deep Learning Foundations**.
>
> 9.1 用框架搭了 MLP，但**它内部怎么训练的？** 这一课用**纯 NumPy 手写一遍**：前向传播、**反向传播(backprop)**、梯度下降。反向传播是深度学习的"心脏"——它就是**链式法则**(0.8)的系统应用。亲手推一遍、写一遍，是真正搞懂深度学习的唯一途径，也是面试白板题的高频区。
> 9.1 built an MLP with a framework, but **how does it train inside?** This lesson hand-codes it in **pure NumPy**: the forward pass, **backpropagation**, and gradient descent. Backprop is deep learning's "heart" — a systematic application of the **chain rule** (0.8). Deriving and coding it once is the only way to truly understand DL, and a whiteboard-interview staple.
>
> 💼 **实战/面试视角**："手推反向传播 / 链式法则 / 梯度怎么流" 是深度学习白板题最高频区。
> 💼 **Practical/interview angle:** "derive backprop / chain rule / gradient flow" — the most common DL whiteboard area.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{z}^{(l)}=\mathbf{a}^{(l-1)}\mathbf{W}^{(l)}+\mathbf{b}^{(l)}$ —— 第 $l$ 层的加权输入 / pre-activation
> - $\mathbf{a}^{(l)}=\sigma(\mathbf{z}^{(l)})$ —— 第 $l$ 层激活输出 / activation
> - $\delta^{(l)}$ —— 第 $l$ 层的误差(损失对 $\mathbf{z}^{(l)}$ 的梯度)/ layer error

> 💡 **面试相关 / Interview-relevant**
> - "反向传播是什么 / 和链式法则的关系"（出镜率 ★★★★★）
> - "前向传播 vs 反向传播分别在算什么"（★★★★★）
> - "为什么要缓存前向的中间值"（★★★★）
> - "梯度检查怎么做"（★★★）
> - "softmax+交叉熵的梯度为什么是 (p-y)"（★★★★）

---

## 学习目标 / Learning Objectives

1. 推导并实现**前向传播**（一层层算激活）。
   Derive and implement the **forward pass**.
2. 推导并实现**反向传播**（链式法则反向传梯度）。
   Derive and implement **backpropagation** (chain rule, backward).
3. 用**梯度检查**验证手算梯度的正确性。
   Verify hand-derived gradients with **gradient checking**.
4. 用 mini-batch SGD 训练，在 Digits 上达到高准确率。
   Train with mini-batch SGD, reaching high accuracy on Digits.
5. 理解 softmax+交叉熵的梯度恰是 $(p-y)$。
   Understand softmax+cross-entropy's gradient is exactly $(p-y)$.

## 目录 / TOC
1. [先建直觉：前向 + 反向 ⭐](#1)
2. [反向传播的数学 ⭐](#2)
3. [🔢 数据 + 从零网络 ⭐](#3)
4. [梯度检查 ⭐](#4)
5. [训练 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：前向 + 反向 ⭐ / Intuition: Forward + Backward

训练神经网络就是**调权重让损失变小**，靠梯度下降——核心是算出"损失对每个权重的梯度"。一个网络有成千上万个权重，怎么高效算所有梯度？答案是**反向传播**，分两趟：
Training a net means **adjusting weights to lower the loss** via gradient descent — the core is computing "the gradient of the loss w.r.t. each weight". A net has thousands of weights; how to compute all gradients efficiently? The answer is **backpropagation**, in two passes:

- **前向传播(forward)**：从输入一层层往后算，得到每层的激活和最终预测、损失。**沿途缓存中间值**（后面反向要用）。
  **Forward pass:** compute layer by layer from input to the prediction and loss. **Cache intermediate values** along the way (needed in the backward pass).
- **反向传播(backward)**：从损失出发，用**链式法则**(0.8)把梯度**一层层往前传**，得到损失对每个权重的偏导。

  **Backward pass:** starting from the loss, use the **chain rule** (0.8) to propagate gradients **backward layer by layer**, giving the partial of the loss w.r.t. every weight.

**为什么必须缓存前向值？**（面试常问）因为反向算梯度时要用到前向的激活值（如 $\frac{\partial L}{\partial W}$ 里含输入激活）。反向传播之所以高效，正是因为它**复用前向算过的量**，避免重复计算——这就是它比"对每个权重单独求数值导数"快无数倍的原因。
**Why cache forward values?** (often asked) Because computing gradients backward reuses forward activations (e.g. $\frac{\partial L}{\partial W}$ contains the input activation). Backprop is efficient precisely because it **reuses forward quantities**, avoiding recomputation — why it's vastly faster than numerically differentiating each weight separately.


<a id="2"></a>
## 2. 反向传播的数学 ⭐ / The Math of Backprop

以一个两层 MLP（输入 → 隐藏(ReLU) → 输出(softmax)）+ 交叉熵损失为例。
Take a two-layer MLP (input → hidden(ReLU) → output(softmax)) with cross-entropy loss.

**前向 / Forward:**
$$\mathbf{z}_1 = \mathbf{X}\mathbf{W}_1+\mathbf{b}_1,\quad \mathbf{a}_1=\text{ReLU}(\mathbf{z}_1),\quad \mathbf{z}_2=\mathbf{a}_1\mathbf{W}_2+\mathbf{b}_2,\quad \mathbf{p}=\text{softmax}(\mathbf{z}_2)$$

**反向 / Backward**（核心是逐层算"误差" $\delta$，再由它得到权重梯度）：
**Backward** (the core is computing the per-layer "error" $\delta$, then weight gradients from it):
- 输出层：softmax+交叉熵的梯度极简（5.1/5.2 见过）：$\delta_2 = \mathbf{p}-\mathbf{y}$（预测概率减真实 one-hot）。
  Output layer: softmax+CE gives the clean $\delta_2 = \mathbf{p}-\mathbf{y}$ (seen in 5.1/5.2).
- 权重梯度：$\frac{\partial L}{\partial \mathbf{W}_2}=\mathbf{a}_1^\top\delta_2$，$\frac{\partial L}{\partial \mathbf{b}_2}=\sum\delta_2$。
  Weight grads: $\frac{\partial L}{\partial \mathbf{W}_2}=\mathbf{a}_1^\top\delta_2$, $\frac{\partial L}{\partial \mathbf{b}_2}=\sum\delta_2$.
- 误差往前传（链式法则 + ReLU 的导数 = [z>0]）：$\delta_1 = (\delta_2\mathbf{W}_2^\top)\odot[\mathbf{z}_1>0]$。
  Propagate the error back (chain rule + ReLU's derivative = [z>0]): $\delta_1 = (\delta_2\mathbf{W}_2^\top)\odot[\mathbf{z}_1>0]$.
- 再得 $\frac{\partial L}{\partial \mathbf{W}_1}=\mathbf{X}^\top\delta_1$，$\frac{\partial L}{\partial \mathbf{b}_1}=\sum\delta_1$。
  Then $\frac{\partial L}{\partial \mathbf{W}_1}=\mathbf{X}^\top\delta_1$, $\frac{\partial L}{\partial \mathbf{b}_1}=\sum\delta_1$.

注意模式：**每层的权重梯度 = (该层输入)ᵀ × (该层误差 δ)**；误差通过 $\mathbf{W}^\top$ 往前传。这个模式对任意层数都成立——这就是反向传播的全部。
Notice the pattern: **each layer's weight gradient = (layer input)ᵀ × (layer error δ)**; the error propagates back through $\mathbf{W}^\top$. This pattern holds for any depth — that's all of backprop.


<a id="3"></a>
## 3. 数据 + 从零网络 ⭐ / Data & The From-scratch Net

用 **Digits**（8×8 手写数字，10 类）。把上面的前向/反向**逐行翻译成 NumPy**。读代码时对照公式：`z1/a1/z2/probs` 是前向缓存，`d2/d1` 是误差 δ。
Using **Digits** (8×8, 10 classes). We translate the forward/backward above **line by line into NumPy**. Read alongside the formulas: `z1/a1/z2/probs` are the forward cache, `d2/d1` are the errors δ.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

digits = load_digits()
X = digits.data / 16.0                                    # 归一化 / scale to [0,1]
Y = np.eye(10)[digits.target]                            # one-hot 标签 (n, 10)
X_tr, X_te, Y_tr, Y_te, y_tr, y_te = train_test_split(X, Y, digits.target, test_size=0.3,
                                                       stratify=digits.target, random_state=0)

def relu(z): return np.maximum(0, z)
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)                 # 减最大值防溢出(5.2)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

class TwoLayerNet:
    def __init__(self, d_in, d_hid, d_out, seed=0):
        r = np.random.default_rng(seed)
        # He 初始化(9.9): ×√(2/fan_in), 配 ReLU / He init for ReLU
        self.W1 = r.normal(0, np.sqrt(2/d_in), (d_in, d_hid)); self.b1 = np.zeros(d_hid)
        self.W2 = r.normal(0, np.sqrt(2/d_hid), (d_hid, d_out)); self.b2 = np.zeros(d_out)
    def forward(self, X):
        self.X = X
        self.z1 = X @ self.W1 + self.b1                  # 隐藏层加权输入
        self.a1 = relu(self.z1)                          # 隐藏层激活
        self.z2 = self.a1 @ self.W2 + self.b2           # 输出层加权输入
        self.probs = softmax(self.z2)                    # 预测概率
        return self.probs
    def backward(self, Y):
        n = len(Y)
        d2 = (self.probs - Y) / n                        # 输出层误差 δ2 = (p-y)/n (softmax+CE)
        self.dW2 = self.a1.T @ d2                        # 权重梯度 = 该层输入ᵀ × δ
        self.db2 = d2.sum(0)
        d1 = (d2 @ self.W2.T) * (self.z1 > 0)            # 误差往前传: ×W2ᵀ, 再 ×ReLU'(=[z>0])
        self.dW1 = self.X.T @ d1                         # 同样的模式
        self.db1 = d1.sum(0)
    def step(self, lr):                                  # 梯度下降更新所有参数
        self.W1 -= lr*self.dW1; self.b1 -= lr*self.db1
        self.W2 -= lr*self.dW2; self.b2 -= lr*self.db2

net = TwoLayerNet(64, 64, 10)
net.forward(X_tr[:5]); net.backward(Y_tr[:5])
print("网络搭好; 前向缓存形状:", {"z1": net.z1.shape, "probs": net.probs.shape})
print("梯度形状:", {"dW1": net.dW1.shape, "dW2": net.dW2.shape}, "(应与 W1/W2 一致)")


<a id="4"></a>
## 4. 梯度检查 ⭐ / Gradient Checking

手推梯度容易出错（一个符号、一个转置就错）。**梯度检查(gradient checking)** 是验证它的标准方法：用**数值微分**（把某个权重 $w$ 加/减一个极小 $\epsilon$，看损失变化）算近似梯度 $\frac{L(w+\epsilon)-L(w-\epsilon)}{2\epsilon}$，和反向传播算的解析梯度对比。两者应**几乎相等**（相对误差 < $10^{-6}$）。这是写自定义层时的必备调试技能。
Hand-derived gradients are error-prone (one wrong sign or transpose). **Gradient checking** is the standard verification: compute a numerical gradient via finite differences ($\frac{L(w+\epsilon)-L(w-\epsilon)}{2\epsilon}$) and compare to the analytic gradient from backprop. They should be **nearly equal** (relative error < $10^{-6}$). An essential debugging skill when writing custom layers.


In [ ]:
def cross_entropy(probs, Y):
    return -np.sum(Y * np.log(probs + 1e-12)) / len(Y)   # 交叉熵损失

net = TwoLayerNet(64, 32, 10)
Xb, Yb = X_tr[:20], Y_tr[:20]
net.forward(Xb); net.backward(Yb)                        # 解析梯度(反向传播算的)
analytic = net.dW1.copy()

# 数值梯度: 对 W1 的几个元素, 用中心差分近似 / numerical gradient via central differences
eps = 1e-5; numeric = np.zeros_like(net.W1)
for i, j in [(0,0),(5,10),(20,20),(63,31)]:              # 抽查几个权重
    orig = net.W1[i,j]
    net.W1[i,j] = orig + eps; net.forward(Xb); Lp = cross_entropy(net.probs, Yb)
    net.W1[i,j] = orig - eps; net.forward(Xb); Lm = cross_entropy(net.probs, Yb)
    net.W1[i,j] = orig
    numeric[i,j] = (Lp - Lm) / (2*eps)                   # 中心差分

for i, j in [(0,0),(5,10),(20,20),(63,31)]:
    a, nu = analytic[i,j], numeric[i,j]
    rel = abs(a-nu)/(abs(a)+abs(nu)+1e-12)               # 相对误差
    print(f"W1[{i},{j}]: 解析={a:+.6f}, 数值={nu:+.6f}, 相对误差={rel:.2e} {'✓' if rel<1e-5 else '✗'}")
print("\n相对误差 < 1e-5 → 反向传播实现正确(解析梯度=数值梯度)")


<a id="5"></a>
## 5. 训练 + 小结 ⭐ / Training & Summary

梯度对了，就能训练。用 **mini-batch SGD**：每次取一小批数据做前向+反向+更新（比全量梯度下降快、比单样本稳）。看损失下降、准确率上升。
With correct gradients, we train. Use **mini-batch SGD**: each step takes a small batch for forward+backward+update (faster than full-batch, steadier than single-sample). Watch the loss drop and accuracy rise.


In [ ]:
net = TwoLayerNet(64, 64, 10, seed=0)
n, batch = len(X_tr), 64
losses, accs = [], []
for epoch in range(60):
    perm = rng.permutation(n)                            # 每轮打乱 / shuffle each epoch
    for s in range(0, n, batch):
        idx = perm[s:s+batch]
        net.forward(X_tr[idx]); net.backward(Y_tr[idx])  # 一个 mini-batch: 前向+反向
        net.step(lr=0.5)                                 # 梯度下降更新
    net.forward(X_tr); losses.append(cross_entropy(net.probs, Y_tr))
    accs.append((net.forward(X_te).argmax(1) == y_te).mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(losses); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("训练损失"); axes[0].set_title("损失下降")
axes[1].plot(accs); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("test 准确率"); axes[1].set_title("准确率上升")
plt.tight_layout(); plt.show()
print(f"纯 NumPy 两层网络 final test 准确率: {accs[-1]:.3f}")
print("没有任何框架, 纯手写前向+反向+SGD, 也能训出好模型 — 这就是深度学习的底层机制")


```
训练 = 梯度下降调权重降损失; 关键是高效算所有梯度 → 反向传播
两趟: 前向(逐层算激活+损失, 缓存中间值) → 反向(链式法则把梯度逐层往前传)
反向核心模式: 每层权重梯度 = (该层输入)ᵀ × (该层误差 δ); δ 经 Wᵀ 往前传
  输出层(softmax+CE): δ = p - y (极简); ReLU 反向: ×[z>0]
为什么缓存前向值: 反向复用前向激活 → 避免重复计算 → 比逐权重数值求导快无数倍
梯度检查: 数值梯度 ≈ 解析梯度(相对误差<1e-6) → 验证 backprop 正确
mini-batch SGD: 小批前向+反向+更新; 快且稳
```

### 💡 面试速查 / Interview cheat-sheet
1. **反向传播 = 链式法则的系统应用**, 从损失往输入逐层传梯度。
   Backprop = systematic chain rule, propagating gradients from loss to input layer by layer.
2. **前向缓存中间值, 反向复用** → 高效(比逐权重数值导数快无数倍)。
   Forward caches values, backward reuses them → efficient (vastly faster than per-weight numerical diff).
3. **每层权重梯度 = 输入ᵀ × 误差δ**; δ 经 Wᵀ 往前传。
   Each weight grad = inputᵀ × error δ; δ propagates back through Wᵀ.
4. **softmax+交叉熵的梯度 = p-y**(干净, 故输出层用它)。
   softmax+CE gradient = p-y (clean; hence used at the output).
5. **梯度检查**(数值≈解析)是写自定义层的必备验证。
   Gradient checking (numerical ≈ analytic) is essential for custom layers.

### 下一节 / Next
**9.3 PyTorch 入门**——手写过 backprop 后, 看框架怎么用 **autograd 自动微分**替你做这一切: tensor、nn.Module、DataLoader、训练循环。
**9.3 PyTorch Basics** — having hand-coded backprop, see how a framework does it all via **autograd**: tensors, nn.Module, DataLoader, and the training loop.
